# Step 3: Delayed Re-annotation Review

这个 notebook 用于导出 delayed re-annotation 子集、接收第二轮标签，并自动检查哪些 `source set` 和 `pairing` 需要修订。

In [1]:
import csv
import json
from collections import Counter
from pathlib import Path


## 1. Configure paths and the delayed-review subset

In [2]:
PROJECT_ROOT = Path('/content/2026_SelectTransfer')
if not PROJECT_ROOT.exists():
    if Path.cwd().name == 'notebooks':
        PROJECT_ROOT = Path.cwd().resolve().parents[0]
    else:
        PROJECT_ROOT = Path('/Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer')

PILOT_DIR = PROJECT_ROOT / 'pilot'
RESULTS_DIR = PROJECT_ROOT / 'results' / '03_delayed_reannotation_review'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TAXONOMY_PATH = PILOT_DIR / 'taxonomy.csv'
SOURCE_SETS_PATH = PILOT_DIR / 'source_sets.csv'
PAIRING_PATH = PILOT_DIR / 'pairing_table.csv'

DELAYED_TASK_IDS = [
    'wiki_dev_0123',
    'wiki_dev_12298',
    'wiki_dev_2639',
    'wiki_dev_1379',
    'wiki_dev_10727',
]

REANNOTATION_TEMPLATE_PATH = RESULTS_DIR / 'delayed_reannotation_template.csv'
REANNOTATION_FILLED_PATH = RESULTS_DIR / 'delayed_reannotation_filled.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RESULTS_DIR =', RESULTS_DIR)


PROJECT_ROOT = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer
RESULTS_DIR = /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review


## 2. Load current working tables

In [3]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

taxonomy_rows = read_csv(TAXONOMY_PATH)
source_set_rows = read_csv(SOURCE_SETS_PATH)
pairing_rows = read_csv(PAIRING_PATH)

print('taxonomy rows:', len(taxonomy_rows))
print('source set rows:', len(source_set_rows))
print('pairing rows:', len(pairing_rows))
print()
label_counts = Counter(r['reasoning_label'] for r in taxonomy_rows if r['keep_drop'] == 'keep')
print('current keep label counts:', dict(label_counts))


taxonomy rows: 35
source set rows: 2
pairing rows: 10

current keep label counts: {'bridge': 14, 'comparison': 21}


## 3. Export the delayed re-annotation template

In [4]:
template_rows = []
for row in taxonomy_rows:
    if row['task_id'] not in DELAYED_TASK_IDS:
        continue
    template_rows.append({
        'task_id': row['task_id'],
        'dataset': row['dataset'],
        'question': row['question'],
        'answer': row['answer'],
        'first_pass_reasoning_label': row['reasoning_label'],
        'first_pass_keep_drop': row['keep_drop'],
        'first_pass_note': row['note'],
        'second_pass_reasoning_label': '',
        'second_pass_keep_drop': '',
        'second_pass_note': '',
    })

template_fieldnames = [
    'task_id',
    'dataset',
    'question',
    'answer',
    'first_pass_reasoning_label',
    'first_pass_keep_drop',
    'first_pass_note',
    'second_pass_reasoning_label',
    'second_pass_keep_drop',
    'second_pass_note',
]

write_csv(REANNOTATION_TEMPLATE_PATH, template_rows, template_fieldnames)
print(f'Wrote template to: {REANNOTATION_TEMPLATE_PATH}')
for row in template_rows:
    print(row['task_id'], row['first_pass_reasoning_label'], row['first_pass_keep_drop'])


Wrote template to: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review/delayed_reannotation_template.csv
wiki_dev_0123 comparison keep
wiki_dev_10727 comparison keep
wiki_dev_12298 comparison keep
wiki_dev_2639 bridge keep
wiki_dev_1379 bridge keep


## 4. Fill `second_pass_*` columns, then save as `delayed_reannotation_filled.csv`

先把模板复制成 filled 文件，再手动填写第二轮标注。下面这个 cell 只做存在性检查。

In [5]:
if not REANNOTATION_FILLED_PATH.exists():
    print('No filled file yet.')
    print('Next step: copy delayed_reannotation_template.csv to delayed_reannotation_filled.csv and fill second_pass_* columns.')
else:
    print(f'Found filled file: {REANNOTATION_FILLED_PATH}')


No filled file yet.
Next step: copy delayed_reannotation_template.csv to delayed_reannotation_filled.csv and fill second_pass_* columns.


## 5. Load second-pass annotations and compare against the first pass

In [7]:
if not REANNOTATION_FILLED_PATH.exists():
    raise FileNotFoundError(f'Missing filled review file: {REANNOTATION_FILLED_PATH}')

comparison_rows = read_csv(REANNOTATION_FILLED_PATH)
required_cols = {
    'task_id',
    'first_pass_reasoning_label',
    'first_pass_keep_drop',
    'second_pass_reasoning_label',
    'second_pass_keep_drop',
}
missing = required_cols.difference(comparison_rows[0].keys()) if comparison_rows else required_cols
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

for row in comparison_rows:
    row['label_changed'] = str(row['first_pass_reasoning_label'] != row['second_pass_reasoning_label'])
    row['keep_drop_changed'] = str(row['first_pass_keep_drop'] != row['second_pass_keep_drop'])
    row['any_change'] = str((row['first_pass_reasoning_label'] != row['second_pass_reasoning_label']) or (row['first_pass_keep_drop'] != row['second_pass_keep_drop']))

for row in comparison_rows:
    print(row['task_id'], 'label_changed=', row['label_changed'], 'keep_drop_changed=', row['keep_drop_changed'])


wiki_dev_0123 label_changed= False keep_drop_changed= False
wiki_dev_10727 label_changed= False keep_drop_changed= False
wiki_dev_12298 label_changed= False keep_drop_changed= False
wiki_dev_2639 label_changed= False keep_drop_changed= False
wiki_dev_1379 label_changed= False keep_drop_changed= False


## 6. Summarize stability

In [8]:
summary = {
    'reviewed_tasks': len(comparison_rows),
    'label_changed': sum(row['label_changed'] == 'True' for row in comparison_rows),
    'keep_drop_changed': sum(row['keep_drop_changed'] == 'True' for row in comparison_rows),
    'any_change': sum(row['any_change'] == 'True' for row in comparison_rows),
}
print(summary)

stability_report_path = RESULTS_DIR / 'stability_summary.json'
stability_report_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Wrote summary to: {stability_report_path}')


{'reviewed_tasks': 5, 'label_changed': 0, 'keep_drop_changed': 0, 'any_change': 0}
Wrote summary to: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review/stability_summary.json


## 7. Build the updated taxonomy preview (without overwriting the main file yet)

In [9]:
review_map = {row['task_id']: row for row in comparison_rows}
taxonomy_updated_rows = []
for row in taxonomy_rows:
    new_row = dict(row)
    review = review_map.get(row['task_id'])
    if review is not None:
        new_row['reasoning_label'] = review['second_pass_reasoning_label']
        new_row['keep_drop'] = review['second_pass_keep_drop']
        if review.get('second_pass_note', '').strip():
            new_row['note'] = review['second_pass_note']
    taxonomy_updated_rows.append(new_row)

updated_preview_path = RESULTS_DIR / 'taxonomy_after_delayed_review_preview.csv'
write_csv(updated_preview_path, taxonomy_updated_rows, taxonomy_rows[0].keys())
print(f'Wrote updated taxonomy preview to: {updated_preview_path}')


Wrote updated taxonomy preview to: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review/taxonomy_after_delayed_review_preview.csv


## 8. Detect affected source sets

In [10]:
changed_task_ids = {row['task_id'] for row in comparison_rows if row['any_change'] == 'True'}

def parse_members(cell):
    if not cell or not str(cell).strip():
        return []
    return [x.strip() for x in str(cell).split('|') if x.strip()]

affected_source_sets = []
for row in source_set_rows:
    members = parse_members(row['member_task_ids'])
    touched = [m for m in members if m in changed_task_ids]
    if touched:
        affected_source_sets.append({
            'source_set_id': row['source_set_id'],
            'cluster': row['cluster'],
            'touched_members': '|'.join(touched),
        })

affected_source_sets_path = RESULTS_DIR / 'affected_source_sets.csv'
if affected_source_sets:
    write_csv(affected_source_sets_path, affected_source_sets, affected_source_sets[0].keys())
else:
    write_csv(affected_source_sets_path, [], ['source_set_id', 'cluster', 'touched_members'])
print(f'Wrote affected source set report to: {affected_source_sets_path}')
for row in affected_source_sets:
    print(row)


Wrote affected source set report to: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review/affected_source_sets.csv


## 9. Detect affected pairing rows

In [11]:
affected_pair_rows = [row for row in pairing_rows if row['target_task_id'] in changed_task_ids]
affected_pair_rows_path = RESULTS_DIR / 'affected_pairing_rows.csv'
if affected_pair_rows:
    write_csv(affected_pair_rows_path, affected_pair_rows, pairing_rows[0].keys())
else:
    write_csv(affected_pair_rows_path, [], pairing_rows[0].keys())
print(f'Wrote affected pairing rows to: {affected_pair_rows_path}')
for row in affected_pair_rows:
    print(row['target_task_id'], row['target_cluster'])


Wrote affected pairing rows to: /Users/mac/studyspace/Knowledge-Markdown/capabilities/memory/2026_SelectTransfer/results/03_delayed_reannotation_review/affected_pairing_rows.csv


## 10. Optional: overwrite `pilot/taxonomy.csv` only after manual confirmation

默认不自动覆盖主工作表。只有在你确认 second-pass 结果就是新的 working version 时，才把 `APPLY_TO_MAIN_TAXONOMY` 改成 `True`。

In [12]:
APPLY_TO_MAIN_TAXONOMY = False

if APPLY_TO_MAIN_TAXONOMY:
    write_csv(TAXONOMY_PATH, taxonomy_updated_rows, taxonomy_rows[0].keys())
    print(f'Updated main taxonomy file: {TAXONOMY_PATH}')
else:
    print('Dry run only. Main taxonomy file was not changed.')


Dry run only. Main taxonomy file was not changed.


## 11. Go / No-Go suggestion

这个 cell 只给机械建议，不替代人工判断。

In [13]:
if summary['any_change'] == 0:
    print('Suggested status: GO')
    print('- delayed re-annotation is stable on the selected boundary tasks')
    print('- source sets and pairing table can move toward freeze review')
else:
    print('Suggested status: REVIEW')
    print('- some delayed re-annotation labels changed')
    print('- inspect affected_source_sets.csv and affected_pairing_rows.csv before freezing Round 1 inputs')


Suggested status: GO
- delayed re-annotation is stable on the selected boundary tasks
- source sets and pairing table can move toward freeze review
